In [11]:
import torch 
import torch.nn as nn
import torch.optim as optim 

import torchvision #offical lib for cv
from torchvision.datasets import CIFAR10

In [2]:
#Datasets & DataLoader

from torch.utils.data import DataLoader
import torchvision.transforms as transforms #to transform images 

#image = scale(0,1) , Normalize = (1,-1)


transform = transforms.Compose( [                  #transformation to apply on every image 
    transforms.ToTensor(), #convert every img to pytorch transformer &  automatically perform scaling
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #to normalize data to get -1 and 1 , std value is listed here
    
]) #operations are defined here not execute yet 

train_set = CIFAR10(root = "./data", train = True , download = True, transform = transform) #transform is the transformation that will take place 
test_set = CIFAR10(root = "./data", train = False , download = True, transform = transform)

#CNN for image classification
#i/p image size is 32x32x3 , o/p will be 10 neurons as we have 10 classes

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [37:32<00:00, 75.7kB/s]


In [12]:
trainloader = DataLoader(train_set,batch_size = 64, shuffle = True) 
testloader = DataLoader(test_set, batch_size = 64)

### Build the CNN

In [40]:
#code will be similar to ANN

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
#first layer with convulation layer + ReLU  and self pooling
        self.Conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size = 3, padding = 1), #32 is the no of filters cal through formula, get doubled after every layer  
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kERNAL = 2 stride = 2

         
            nn.Conv2d(32,64,kernel_size = 3, padding = 1), #32 is the no of filters cal through formula, get doubled after every layer  
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kERNAL = 2 stride = 2

           
            nn.Conv2d(64,128,kernel_size = 3, padding = 1), #32 is the no of filters cal through formula, get doubled after every layer  128
            nn.ReLU(),
            nn.MaxPool2d(2,2) # kERNAL = 2 stride = 2
)
        #data is flattened
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.Conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)
        return x

In [41]:
model = CNN()

In [42]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [47]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad() #stops gradient from piling up start with zero at the end of loop 

        output = model.forward(images) #Forward Prop
        loss = criterion(output,labels) #Loss Function
        loss.backward() #BackProp
        optimizer.step() #Update Para

        epoch_training_loss += loss.item() #to convert into py 

    print(f"epoch = {epoch +1}/{epochs}, Loss = {epoch_training_loss/len(trainloader)}")
    

epoch = 1/10, Loss = 0.9303402831334897
epoch = 2/10, Loss = 0.7554418078011564
epoch = 3/10, Loss = 0.6283476990278419
epoch = 4/10, Loss = 0.5232406115669119
epoch = 5/10, Loss = 0.4319447242771573
epoch = 6/10, Loss = 0.3483064760408743
epoch = 7/10, Loss = 0.2785097330027377
epoch = 8/10, Loss = 0.2145128447914977
epoch = 9/10, Loss = 0.17261749618422345
epoch = 10/10, Loss = 0.1366770485763812


### Evaluation

In [48]:
#evaluation
correct_labels = 0
total_labels = 0 

with torch.no_grad(): #no backward prop for updating values
    for images, labels in testloader:
        outputs = model.forward(images) #Forward prop
        _, predicted = torch.max(outputs,1) #to get correct labels 

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels * 100}")

Accuracy = 75.55
